In [18]:
from datetime import time
class Employee:
    def __init__(self, first_name,availability):
        self.first_name = first_name
        self.availability = availability

employee1 = Employee(
    first_name="John",
    availability={
        "Monday": (time(8, 0), time(18, 0)),
        "Tuesday": (time(8, 0), time(18, 0)),
        "Wednesday": (time(8, 0), time(18, 0)),
        "Thursday": (time(8, 0), time(18, 0)),
        "Friday": (time(8, 0), time(18, 0)),
        "Saturday": (time(8, 0), time(18, 0)),
        "Sunday": (time(8, 0), time(18, 0)),
    }
)

employee2 = Employee(
    first_name="Jane",
    availability={
        "Monday": (time(8, 0), time(18, 0)),
        "Tuesday": (time(8, 0), time(18, 0)),
        "Wednesday": (time(8, 0), time(18, 0)),
        "Thursday": (time(8, 0), time(18, 0)),
        "Friday": (time(8, 0), time(18, 0)),
        "Saturday": (time(8, 0), time(18, 0)),
        "Sunday": (time(8, 0), time(18, 0)),
    }
)

employee3 = Employee(
    first_name="Joe",
    availability={
        "Tuesday": (time(8, 0), time(18, 0)),
        "Wednesday": (time(8, 0), time(18, 0)),
        "Thursday": (time(8, 0), time(18, 0)),
        "Friday": (time(8, 0), time(18, 0)),
        "Saturday": (time(8, 0), time(18, 0)),
        "Sunday": (time(8, 0), time(18, 0)),
    }
)

employees = [employee1, employee2,employee3]

In [19]:
class Shift:
    def __init__(self, name, start_time, end_time, shift_map):
        self.name = name
        self.start_time = start_time
        self.end_time = end_time
        self.shift_map = shift_map

# Example usage:
shift1 = Shift(
    name="Morning Shift",
    start_time="08:00",
    end_time="12:00",
    shift_map={
        "Monday": True,
        "Tuesday": True,
        "Wednesday": True,
        "Thursday": True,
        "Friday": True,
        "Saturday": True,
        "Sunday": False,
    }
)

shift2 = Shift(
    name="Evening Shift",
    start_time="14:00",
    end_time="18:00",
    shift_map={
        "Monday": True,
        "Tuesday": True,
        "Wednesday": True,
        "Thursday": True,
        "Friday": True,
        "Saturday": True,
        "Sunday": False,
    }
)

shift3 = Shift(
    name="Lunch Shift",
    start_time="12:00",
    end_time="15:00",
    shift_map={
        "Monday": False,
        "Tuesday": False,
        "Wednesday": False,
        "Thursday": False,
        "Friday": True,
        "Saturday": True,
        "Sunday": True,
    }
)

shifts = [shift1, shift2, shift3]


In [20]:
import pandas as pd

employee_names = [employee.first_name for employee in employees]
availability_data = [
    [1 if employee.availability.get(day) else 0 for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]]
    for employee in employees
]

availability_df = pd.DataFrame(availability_data, index=employee_names, columns=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])

print(availability_df)

      Monday  Tuesday  Wednesday  Thursday  Friday  Saturday  Sunday
John       1        1          1         1       1         1       1
Jane       1        1          1         1       1         1       1
Joe        0        1          1         1       1         1       1


In [21]:
shift_availability_data = []
for shift in shifts:
    shift_row = []
    for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]:
        if shift.shift_map.get(day):
            shift_row.append(f"{shift.start_time}-{shift.end_time}")
        else:
            shift_row.append('x')
    shift_availability_data.append(shift_row)

shift_names = [shift.name for shift in shifts]

shift_availability_df = pd.DataFrame(shift_availability_data, index=shift_names, columns=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])

shift_availability_df

,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
Morning Shift,08:00-12:00,08:00-12:00,08:00-12:00,08:00-12:00,08:00-12:00,08:00-12:00,x
Evening Shift,14:00-18:00,14:00-18:00,14:00-18:00,14:00-18:00,14:00-18:00,14:00-18:00,14:00-18:00
Lunch Shift,x,x,x,x,12:00-15:00,12:00-15:00,12:00-15:00


# PuLP

In [22]:
from pulp import LpProblem, LpVariable, lpSum, value, LpBinary

class ShiftScheduler:
    def __init__(self, employees, shifts):
        self.employees = employees
        self.shifts = shifts
        self.assignments = None

    def create_lp_problem(self):
        # Create LP problem
        prob = LpProblem("Shift_Scheduling", sense=1)  # sense=1 for minimization

        # Decision variables
        assignments = LpVariable.dicts("Assign", ((employee, shift) for employee in self.employees for shift in self.shifts), 0, 1, LpBinary)

        # Objective function (minimize the total number of shifts assigned)
        prob += lpSum(assignments[employee, shift] for employee in self.employees for shift in self.shifts)

        # Constraints
        # Each shift must be covered
        for shift in self.shifts:
            prob += lpSum(assignments[employee, shift] for employee in self.employees) >= 1

        for employee in self.employees:
            for day in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]:
                shifts_on_day = [shift for shift in self.shifts if self.shifts[shift].shift_map[day]]
                prob += lpSum(assignments[employee, shift] for shift in shifts_on_day) <= 1
        # Each employee can only work during their available times
        #for employee in self.employees:
        #    for shift in self.shifts:
        #        if not self.is_employee_available(employee, shift):
         #           prob += assignments[employee, shift] == 0

        # Each employee can only be assigned a limited number of shifts (adjust as needed)
        #max_shifts_per_employee = 3
        #for employee in self.employees:
        #    prob += lpSum(assignments[employee, shift] for shift in self.shifts) <= max_shifts_per_employee

        self.assignments = assignments
        return prob

    def is_employee_available(self, employee, shift_name):
        # Check if the employee is available during the shift's time
        shift = self.shifts[shift_name]
        employee_availability = self.employees[employee].date_time.get(shift_name, None)
        if employee_availability:
            shift_start, shift_end = shift.start_time, shift.end_time
            return shift_name in self.employees[employee].date_time and shift_start >= employee_availability[0] and shift_end <= employee_availability[1]
        return False

    def solve_schedule(self):
        prob = self.create_lp_problem()
        prob.solve()

        if prob.status == 1:  # Optimal solution found
            self.print_schedule()

    def print_schedule(self):
        days_of_week = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
        for day in days_of_week:
            print(f"\nAssignments for {day}:")
            for employee in self.employees:
                for shift in self.shifts:
                    if self.shifts[shift].shift_map[day] and value(self.assignments[employee, shift]) == 1:
                        print(f"{employee} is assigned to {shift}")

# Example usage:
employee1 = Employee(
    first_name="John",
    last_name="Doe",
    email="john.doe@example.com",
    phone_number="123-456-7890",
    date_time={
        "Monday": ("08:00", "18:00"),
        "Tuesday": ("08:00", "18:00"),
        "Wednesday": ("08:00", "18:00"),
        "Thursday": ("08:00", "18:00"),
        "Friday": ("08:00", "18:00"),
        "Saturday": ("08:00", "18:00"),
        "Sunday": ("08:00", "18:00"),
    }
)

employee2 = Employee(
    first_name="Jane",
    last_name="Smith",
    email="jane.smith@example.com",
    phone_number="987-654-3210",
    date_time={
        "Monday": ("08:00", "18:00"),
        "Tuesday": ("08:00", "18:00"),
        "Wednesday": ("08:00", "18:00"),
        "Thursday": ("08:00", "18:00"),
        "Friday": ("08:00", "18:00"),
        "Saturday": ("08:00", "18:00"),
        "Sunday": ("08:00", "18:00"),
    }
)

shift1 = Shift(
    name="Morning Shift",
    start_time="08:00",
    end_time="12:00",
    shift_map={
        "Monday": True,
        "Tuesday": True,
        "Wednesday": True,
        "Thursday": True,
        "Friday": True,
        "Saturday": False,
        "Sunday": False,
    }
)

shift2 = Shift(
    name="Evening Shift",
    start_time="12:00",
    end_time="14:00",
    shift_map={
        "Monday": True,
        "Tuesday": True,
        "Wednesday": True,
        "Thursday": True,
        "Friday": True,
        "Saturday": True,
        "Sunday": True,
    }
)

# List of employees and shifts
employees = {"John": employee1, "Jane": employee2}
shifts = {"Morning Shift": shift1, "Evening Shift": shift2}

# Create a ShiftScheduler instance and solve the schedule
scheduler = ShiftScheduler(employees, shifts)
scheduler.solve_schedule()


TypeError: __init__() got an unexpected keyword argument 'last_name'